In [1]:
# ============================================================
#  08 POI Demand Layers
#  数据源: 高德 POI 搜索 API (覆盖率远超 OSM)
#  输出: POI 点位 + 网格化需求密度
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import time
import json
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from shapely.prepared import prep as shapely_prep
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")

AMAP_KEY = "305355ae5b6f7ff3201d61cb4ae4b469"

# 高德 POI 大类编码 → 我们的分类
POI_CATEGORIES = {
    "050000": "food",           # 餐饮
    "050500": "cafe",           # 咖啡/茶饮 (子类, 会被 050000 覆盖, 单独抓更精确)
    "060100": "convenience",    # 便利店
    "060400": "supermarket",    # 超市
    "060700": "mall",           # 购物中心/商场
    "090100": "hospital",       # 医院
    "090200": "clinic",         # 诊所
    "141200": "school",         # 学校
    "120000": "office",         # 商务写字楼
    "120300": "residential",    # 住宅区
}

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"AMap Key: ...{AMAP_KEY[-6:]}")
print(f"POI categories: {len(POI_CATEGORIES)}")
print(f"Bbox: {[round(x,3) for x in bbox]}")

/Users/shirly/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMap Key: ...e4b469
POI categories: 10
Bbox: [113.747, 22.397, 114.623, 22.855]


In [ ]:
# ============================================================
#  1. 高德 API 批量抓取 POI
#  按网格扫描深圳, 每个网格 × 每个类别发一次请求
#  高德 polygon 搜索 API: 一次最多返回 25 条, 需分页
#  免费配额: 5000 次/日, 注意控速
# ============================================================

CACHE_FILE = RAW / "amap_poi_cache.json"

def amap_search_poi(polygon_str, typecode, page=1, retries=3):
    """调用高德 POI 搜索 (polygon 范围), 含重试机制"""
    url = "https://restapi.amap.com/v3/place/polygon"
    params = {
        "key": AMAP_KEY,
        "polygon": polygon_str,
        "types": typecode,
        "offset": 25,
        "page": page,
        "extensions": "base",
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                print(f"  Request failed after {retries} retries: {e}")
                return None

def fetch_all_in_cell(polygon_str, typecode, our_type):
    """分页抓取一个网格内某类别的全部 POI"""
    results = []
    page = 1
    while True:
        data = amap_search_poi(polygon_str, typecode, page)
        if data is None or data.get("status") != "1":
            break
        pois = data.get("pois", [])
        for p in pois:
            loc = p.get("location", "")
            if not loc or "," not in loc:
                continue
            lon, lat = loc.split(",")
            results.append({
                "name": p.get("name", ""),
                "poi_type": our_type,
                "amap_type": p.get("type", ""),
                "typecode": typecode,
                "address": p.get("address", ""),
                "lon": float(lon),
                "lat": float(lat),
            })
        if len(pois) < 25:
            break
        page += 1
        time.sleep(0.15)
    return results

# 生成搜索网格 (~1km, 用 0.01°)
SCAN_STEP = 0.01
minx, miny, maxx, maxy = bbox
sz_prep = shapely_prep(shenzhen_geom)

grid_cells = []
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cx, cy = x + SCAN_STEP / 2, y + SCAN_STEP / 2
        if sz_prep.contains(Point(cx, cy)):
            poly_str = f"{x},{y}|{x + SCAN_STEP},{y + SCAN_STEP}"
            grid_cells.append(poly_str)
        y += SCAN_STEP
    x += SCAN_STEP

total_requests = len(grid_cells) * len(POI_CATEGORIES)
print(f"Scan grid: {len(grid_cells)} cells × {len(POI_CATEGORIES)} categories = {total_requests:,} requests")
print(f"Estimated time: {total_requests * 0.2 / 60:.0f} min (with rate limiting)")

# 增量抓取: 加载已有缓存, 跳过已完成的 (typecode, grid_cell) 对
if CACHE_FILE.exists():
    print(f"Loading existing cache from {CACHE_FILE} ...")
    with open(CACHE_FILE) as f:
        all_pois = json.load(f)
    print(f"  Cached POIs: {len(all_pois):,}")
else:
    all_pois = []

# 记录已完成的 typecode (整个类别完成才算)
PROGRESS_FILE = RAW / "amap_progress.json"
if PROGRESS_FILE.exists():
    with open(PROGRESS_FILE) as f:
        done_types = set(json.load(f))
else:
    done_types = set()

remaining = {k: v for k, v in POI_CATEGORIES.items() if k not in done_types}
if not remaining:
    print(f"All categories already fetched. Total: {len(all_pois):,} POIs")
else:
    print(f"Remaining categories: {len(remaining)} / {len(POI_CATEGORIES)}")
    done = 0
    total_remaining = len(grid_cells) * len(remaining)
    for typecode, our_type in remaining.items():
        type_count = 0
        for poly_str in grid_cells:
            pois = fetch_all_in_cell(poly_str, typecode, our_type)
            all_pois.extend(pois)
            type_count += len(pois)
            done += 1
            if done % 200 == 0:
                print(f"  Progress: {done}/{total_remaining} requests, {len(all_pois):,} POIs so far")
                with open(CACHE_FILE, "w") as f:
                    json.dump(all_pois, f, ensure_ascii=False)
            time.sleep(0.12)
        print(f"  {our_type:15s} ({typecode}): {type_count:,} raw POIs")
        done_types.add(typecode)
        with open(PROGRESS_FILE, "w") as f:
            json.dump(list(done_types), f)
        with open(CACHE_FILE, "w") as f:
            json.dump(all_pois, f, ensure_ascii=False)

    print(f"\nTotal raw POIs: {len(all_pois):,}")
    print(f"Saved cache -> {CACHE_FILE}")

Scan grid: 1720 cells × 10 categories = 17,200 requests
Estimated time: 57 min (with rate limiting)
  Progress: 200/17200 requests, 15,922 POIs so far
  Request error: HTTPSConnectionPool(host='restapi.amap.com', port=443): Read timed out. (read timeout=15)


In [ ]:
# ============================================================
#  2. 清洗去重 + 分类
# ============================================================

df = pd.DataFrame(all_pois)
print(f"Raw POIs: {len(df):,}")

# 去重: 相同名称 + 相近坐标 (四舍五入到小数点后 5 位 ≈ 1m)
df["lon_r"] = df["lon"].round(5)
df["lat_r"] = df["lat"].round(5)
df = df.drop_duplicates(subset=["name", "lon_r", "lat_r"]).drop(columns=["lon_r", "lat_r"])
print(f"After dedup: {len(df):,}")

# 构建 GeoDataFrame
poi_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=4326,
)

# 裁剪到深圳边界
poi_gdf = gpd.clip(poi_gdf, shenzhen_geom)
print(f"After clip: {len(poi_gdf):,}")

# 大类归并 (cafe 合并到 food 大类用于计算 demand)
DEMAND_GROUP = {
    "food": "food", "cafe": "food",
    "convenience": "retail", "supermarket": "retail", "mall": "retail",
    "hospital": "medical", "clinic": "medical",
    "school": "education",
    "office": "office",
    "residential": "residential",
}
poi_gdf["demand_group"] = poi_gdf["poi_type"].map(DEMAND_GROUP)

print(f"\n=== POI by type ===")
print(poi_gdf["poi_type"].value_counts().to_string())
print(f"\n=== POI by demand group ===")
print(poi_gdf["demand_group"].value_counts().to_string())

In [ ]:
# ============================================================
#  3. 网格化需求密度 + demand_pressure
#  复用 06 Buildings 的网格大小 (0.005° ≈ 550m)
# ============================================================

GRID_SIZE = 0.005

minx, miny, maxx, maxy = shenzhen_geom.bounds

grid_cells = []
gid = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE, y + GRID_SIZE)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": gid, "geometry": cell})
            gid += 1
        y += GRID_SIZE
    x += GRID_SIZE

grid = gpd.GeoDataFrame(grid_cells, crs=4326)
print(f"Grid: {len(grid):,} cells")

# 空间连接 POI → 网格
joined = gpd.sjoin(poi_gdf, grid[["grid_id", "geometry"]], how="inner", predicate="within")

# 按需求大类分组统计
pivot = joined.groupby(["grid_id", "demand_group"]).size().unstack(fill_value=0)
pivot.columns = [f"{c}_count" for c in pivot.columns]
pivot["poi_total"] = pivot.sum(axis=1)
pivot = pivot.reset_index()

# 合并回网格
grid_demand = grid.merge(pivot, on="grid_id", how="left").fillna(0)

# demand_pressure: 加权综合需求指数
WEIGHTS = {
    "food_count": 0.30,
    "retail_count": 0.25,
    "medical_count": 0.15,
    "office_count": 0.15,
    "education_count": 0.05,
    "residential_count": 0.10,
}

grid_demand["demand_pressure"] = 0
for col, w in WEIGHTS.items():
    if col in grid_demand.columns:
        grid_demand["demand_pressure"] += w * grid_demand[col]

# 归一化到 [0, 1]
dp = grid_demand["demand_pressure"]
dp_max = dp.max()
if dp_max > 0:
    grid_demand["demand_pressure_norm"] = dp / dp_max

non_empty = (grid_demand["poi_total"] > 0).sum()
print(f"Grids with POIs: {non_empty:,} / {len(grid_demand):,}")
print(f"Demand pressure: max={dp.max():.1f}, mean(non-empty)={dp[dp > 0].mean():.1f}")

In [ ]:
# ============================================================
#  4. 保存 + 统计摘要
# ============================================================

# POI 点位
poi_out = poi_gdf[["name", "poi_type", "demand_group", "address", "lon", "lat", "geometry"]]
poi_out.to_file(OUT / "sz_poi.gpkg", driver="GPKG")
print(f"POI      -> data_out/sz_poi.gpkg          ({len(poi_out):,} rows)")

# 需求网格
grid_demand.to_file(OUT / "sz_demand_grid.gpkg", driver="GPKG")
print(f"Grid     -> data_out/sz_demand_grid.gpkg   ({len(grid_demand):,} cells)")

# 统计摘要
print("\n=== POI Summary ===")
print(f"Total POIs: {len(poi_out):,}")
print(poi_out["poi_type"].value_counts().to_string())

print("\n=== Demand Grid Summary (non-empty) ===")
g = grid_demand[grid_demand["poi_total"] > 0]
for col in [c for c in grid_demand.columns if c.endswith("_count")] + ["demand_pressure"]:
    if col in g.columns:
        print(f"  {col:25s}  mean={g[col].mean():8.1f}  max={g[col].max():8.0f}")